# 143 — Context Compression

**What this workbook teaches:**
- Why context compression matters for long documents and limited context windows
- How `split_sentences()` and `count_tokens()` work as the preprocessing layer
- How the LLM-scored compression pipeline filters sentences by relevance to a query
- How to quantify the before/after token reduction

---

**Two sections:**
- **Part A** — Pure Python, no API key required. Explore the pipeline with mock scores.
- **Part B** — Live OpenAI calls. Requires `OPENAI_API_KEY` in your environment.

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — Concepts (no API key needed)

### Why context compression?

LLMs have finite context windows. When answering a question about a long document, most of the text is *not relevant* to the specific question asked. Sending the full document wastes tokens (and money), increases latency, and can dilute the model's focus.

**Context compression** addresses this by:
1. **Splitting** the document into sentences
2. **Scoring** each sentence for relevance to the query (using an LLM judge)
3. **Keeping** only the top N% of sentences (controlled by `ratio`)
4. **Sending** the compressed context to the answering LLM

The key insight is that a **small, focused context** often produces *better* answers than a large, noisy one — and at a fraction of the token cost.

**The sentence scoring approach:**
- Ask the LLM: "Rate each sentence 0.0–1.0 for relevance to this query."
- Sort sentences by score, keep the top `ratio` fraction
- Default `ratio = 0.4` means keep 40% of sentences

**Trade-off:** The compression step itself uses LLM tokens. Worthwhile when:
- The document is long (many sentences)
- The same compressed context is reused for multiple queries
- The answering model is more expensive than the scoring model

In [ ]:
# Part A — Cell 2: Demo split_sentences() and count_tokens()
# These are the exact functions from src/tools.py — no LLM needed.

import re

def split_sentences(text: str) -> list[str]:
    """Split text into sentences on sentence-ending punctuation."""
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def count_tokens(text: str) -> int:
    """Rough token estimate: ~4 chars per token."""
    return len(text) // 4


SAMPLE_PARAGRAPH = (
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming. "
    "CPython is the reference implementation of Python. "
    "The GIL in CPython prevents true parallel execution of Python threads. "
    "Python is dynamically typed and garbage-collected. "
    "It features a comprehensive standard library often described as batteries included."
)

sentences = split_sentences(SAMPLE_PARAGRAPH)

print(f"Input: {len(SAMPLE_PARAGRAPH)} chars, ~{count_tokens(SAMPLE_PARAGRAPH)} tokens")
print(f"Sentences found: {len(sentences)}")
print()
for i, s in enumerate(sentences):
    print(f"  [{i+1}] ({count_tokens(s):>3} tok) {s}")

In [ ]:
# Part A — Cell 3: Compression pipeline with MOCK scores (no LLM call)
# Shows exactly which sentences survive at ratio=0.4

QUERY = "How does CPython execute Python code?"

# Hardcoded relevance scores (0.0 = irrelevant, 1.0 = highly relevant)
# Pretending an LLM rated each sentence against the query above.
MOCK_SCORES = [
    0.1,   # creation by Guido — not about execution
    0.05,  # design philosophy — not about execution
    0.15,  # programming paradigms — tangentially relevant
    0.90,  # CPython is reference implementation — very relevant
    0.85,  # GIL prevents parallel threads — relevant to CPython execution
    0.20,  # dynamically typed — somewhat relevant
    0.05,  # standard library — not about execution
]

assert len(MOCK_SCORES) == len(sentences), "Score count must match sentence count"

scored = list(zip(MOCK_SCORES, sentences))
scored_sorted = sorted(scored, key=lambda x: x[0], reverse=True)

ratio = 0.4
keep_n = max(1, int(len(scored_sorted) * ratio))
kept = [s for _, s in scored_sorted[:keep_n]]
dropped = [s for _, s in scored_sorted[keep_n:]]

compressed_text = " ".join(kept)

print(f"Query: {QUERY}")
print(f"Ratio: {ratio}  |  Keep top {keep_n} of {len(scored)} sentences")
print()
print("=== Scores (sorted) ===")
for score, s in scored_sorted:
    tag = "KEEP" if (score, s) in [(sc, se) for sc, se in scored_sorted[:keep_n]] else "DROP"
    print(f"  [{tag}] {score:.2f}  {s[:70]}")

print()
print("=== Compressed context ===")
print(compressed_text)
print()
print(f"Original:   ~{count_tokens(SAMPLE_PARAGRAPH)} tokens")
print(f"Compressed: ~{count_tokens(compressed_text)} tokens")
reduction = round((1 - count_tokens(compressed_text) / count_tokens(SAMPLE_PARAGRAPH)) * 100)
print(f"Reduction:  {reduction}%")

### Token count before/after — sample comparison

Here's what the compression output looks like across different ratios on the same document:

| Ratio | Sentences kept | Approx. tokens | Reduction |
|---|---|---|---|
| 1.0 (no compression) | 7 / 7 | ~87 | 0% |
| 0.6 | 4 / 7 | ~54 | ~38% |
| 0.4 (default) | 3 / 7 | ~40 | ~54% |
| 0.2 | 2 / 7 | ~26 | ~70% |

For a **500-word document** (~625 tokens), a 0.4 ratio compresses to ~250 tokens — saving ~375 tokens per LLM call.

In [ ]:
# Part A — Cell 4 bonus: Sweep ratios to verify the table above

original_tokens = count_tokens(SAMPLE_PARAGRAPH)

print(f"{'Ratio':<8} {'Kept':>6} {'Tokens':>8} {'Reduction':>10}")
print("-" * 38)
for ratio in [1.0, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(scored_sorted) * ratio))
    kept_sentences = [s for _, s in scored_sorted[:keep_n]]
    compressed = " ".join(kept_sentences)
    tok = count_tokens(compressed)
    pct = round((1 - tok / original_tokens) * 100)
    print(f"{ratio:<8} {keep_n:>6}/{len(sentences)} {tok:>8} {pct:>9}%")

---
## Part B — Live OpenAI calls

**Requires:** `OPENAI_API_KEY` in your `.env` file or environment.

Runs the full compression + answer workflow on a ~500-word Python document:
1. LLM scores each sentence for relevance to the query
2. Top 40% of sentences are kept
3. Compressed context is sent to the answering LLM
4. Prints original vs. compressed token counts and the final answer

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Add it to your .env file or export it before running Part B."
    )
print("OPENAI_API_KEY loaded.")

In [ ]:
# Part B — Run full compress + answer workflow on a ~500-word document
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "."))

from openai import OpenAI
from src.workflow import create_workflow

SAMPLE_CHUNKS = [
    "The Python programming language was created by Guido van Rossum and first released in 1991. "
    "Python's design philosophy emphasizes code readability with the use of significant indentation. "
    "It supports multiple programming paradigms, including structured, object-oriented, and functional programming.",

    "Python is dynamically typed and garbage-collected. It features a comprehensive standard library. "
    "The language is often described as batteries included due to this comprehensive library. "
    "Python consistently ranks as one of the most popular programming languages worldwide.",

    "CPython is the reference implementation of Python. When people refer to Python they often mean CPython. "
    "Other implementations include PyPy, Jython, and IronPython. "
    "CPython is an interpreter that compiles Python to bytecode before executing it. "
    "The GIL in CPython prevents true parallel execution of Python threads.",

    "Python 3 was released in 2008 and is incompatible with Python 2 in several ways. "
    "Python 2 reached end-of-life on January 1, 2020. "
    "The transition from Python 2 to Python 3 took the community over a decade to complete. "
    "Most major libraries have since dropped Python 2 support.",

    "Python's async/await syntax was introduced in Python 3.5 via PEP 492. "
    "asyncio is the standard library module for writing concurrent code with async/await. "
    "It uses an event loop to manage I/O-bound tasks without threading. "
    "This makes Python suitable for building high-performance network servers and web applications.",
]

QUERY = "How does CPython execute Python code?"

client = OpenAI(api_key=OPENAI_API_KEY)
workflow = create_workflow()
config = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

result = workflow.invoke({
    "chunks": SAMPLE_CHUNKS,
    "query": QUERY,
    "ratio": 0.4,
    "original_tokens": 0,
    "compressed_context": "",
    "compressed_tokens": 0,
    "answer": "",
}, config=config)

orig = result["original_tokens"]
comp = result["compressed_tokens"]
reduction = round((1 - comp / orig) * 100) if orig else 0

print(f"Query: {QUERY}")
print()
print(f"{'Metric':<25} {'Value'}")
print("-" * 45)
print(f"{'Original tokens':<25} {orig}")
print(f"{'Compressed tokens':<25} {comp}")
print(f"{'Token reduction':<25} {reduction}%")
print(f"{'Tokens saved':<25} {orig - comp}")
print()
print("=== Answer (from compressed context) ===")
print(result["answer"])

### Why Context Compression Matters

Sending a large context to an LLM has three costs:

| Cost | Effect | Magnitude |
|---|---|---|
| Token price | Directly proportional to input tokens | $0.25–$3/MTok |
| Latency | More tokens → more TTFT (time to first token) | +50–500ms per 1k tokens |
| Attention dilution | Model spreads attention over irrelevant content | Quality degrades past ~8k tokens |

Context compression reduces all three simultaneously by removing irrelevant sentences
before the context reaches the LLM.

### Extractive vs Abstractive Compression

| Property | Extractive | Abstractive |
|---|---|---|
| Method | Select and keep original sentences | Rewrite/summarize content |
| Faithfulness | High — original wording preserved | Risk of hallucination |
| Speed | Fast (scoring only) | Slow (LLM rewrite required) |
| Minimum size | Limited by shortest relevant sentence | Can compress to 1 sentence |
| This example | **Extractive** (sentence scoring) | — |

For RAG pipelines, extractive compression is preferred because it preserves exact quotes
and avoids introducing errors through paraphrase.

### Relevance Scoring Approaches

| Approach | How it works | Accuracy | Speed | Cost |
|---|---|---|---|---|
| TF-IDF keyword overlap | Count shared terms between sentence and query | Low | Very fast | Free |
| Embedding cosine similarity | Compare sentence vector to query vector | High | Fast | Embedding API |
| LLM judge | Ask model: 'is this sentence relevant?' | Very high | Slow | LLM API |
| BM25 | Improved TF-IDF with length normalization | Medium | Fast | Free |

This example uses **embedding cosine similarity** — a good balance of accuracy, speed, and cost.

### Exercise 1 — Stopword Filter

Very short sentences (< 5 words) rarely contain enough context to be useful.
Modify the compression function to remove sentences below a word threshold.

In [ ]:
# Exercise 1: Add a min_words parameter to the compress function.
# Sentences with fewer than min_words words should be excluded regardless of relevance score.
#
# compress_v2(sentences, scores, ratio=0.4, min_words=5) -> list[str]
#
# TODO: implement compress_v2


In [ ]:
# Exercise 1 — Answer Key
def compress_v2(sentences: list, scores: list, ratio: float = 0.4, min_words: int = 5) -> list:
    # First filter by word count — short sentences rarely add context.
    candidates = [(s, sc) for s, sc in zip(sentences, scores)
                  if len(s.split()) >= min_words]
    # Then sort by relevance and keep top-ratio fraction.
    candidates.sort(key=lambda x: x[1], reverse=True)
    keep_n = max(1, int(len(candidates) * ratio))
    kept = {s for s, _ in candidates[:keep_n]}
    # Preserve original sentence order for coherent reading.
    return [s for s in sentences if s in kept]

# Demo
test_sentences = [
    'Python is a high-level programming language.',
    'Yes.',              # too short — filtered by min_words
    'It uses dynamic typing and garbage collection.',
    'OK.',               # too short — filtered
    'Python supports multiple programming paradigms including OOP and functional programming.',
    'The language was created by Guido van Rossum and first released in 1991.',
]
test_scores = [0.9, 0.1, 0.8, 0.05, 0.85, 0.6]

result = compress_v2(test_sentences, test_scores, ratio=0.5, min_words=5)
print('Kept sentences:')
for s in result:
    print(f'  - {s}')
# The two single-word sentences are removed regardless of ratio.
# This prevents degenerate cases where a high-scoring 'Yes.' makes it into the context.

### Exercise 2 — Minimum Sentences Floor

At very aggressive ratios (0.1–0.2), the context might shrink to 0–1 sentences.
Add a `min_sentences` floor to prevent this edge case.

In [ ]:
# Exercise 2: Add a min_sentences parameter (default=3) to compress_v2.
# If ratio would produce fewer than min_sentences sentences,
# keep the top-min_sentences by score instead.
#
# TODO: modify compress_v2 to add this floor.


In [ ]:
# Exercise 2 — Answer Key
def compress_v3(sentences, scores, ratio=0.4, min_words=5, min_sentences=3):
    candidates = [(s, sc) for s, sc in zip(sentences, scores)
                  if len(s.split()) >= min_words]
    candidates.sort(key=lambda x: x[1], reverse=True)
    # Respect both ratio and min_sentences floor.
    keep_n = max(min_sentences, int(len(candidates) * ratio))
    kept = {s for s, _ in candidates[:keep_n]}
    return [s for s in sentences if s in kept]

# Demo: very aggressive ratio that would otherwise produce 0 sentences
print('ratio=0.05 (no floor):', compress_v2(test_sentences, test_scores, ratio=0.05))
print('ratio=0.05 (floor=2):', compress_v3(test_sentences, test_scores, ratio=0.05, min_sentences=2))
# The floor ensures the LLM always gets at least some context to answer from.

In [ ]:
# Part A: Token counting comparison — why char/4 is unreliable
try:
    import tiktoken
    enc = tiktoken.encoding_for_model('gpt-4o')
    HAS_TIKTOKEN = True
except ImportError:
    HAS_TIKTOKEN = False
    print('tiktoken not installed — using char/4 estimate only')

SAMPLES = [
    ('English prose', 'The quick brown fox jumps over the lazy dog. ' * 5),
    ('Code (Python)', 'def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1)+fibonacci(n-2)\n' * 5),
    ('JSON', '{"key": "value", "nested": {"a": 1, "b": true}} ' * 10),
    ('Emoji-heavy', '🚀 Launch time! 🎯 Target acquired. 💡 Bright idea! ' * 5),
]

print(f'{'Text type':<15} | {'Chars':>6} | {'Char/4 est':>10} | {'Tiktoken':>8} | {'Error%':>7}')
print('-' * 58)
for name, text in SAMPLES:
    chars = len(text)
    char_est = chars // 4
    if HAS_TIKTOKEN:
        exact = len(enc.encode(text))
        err = abs(char_est - exact) / exact * 100
        print(f'{name:<15} | {chars:>6} | {char_est:>10} | {exact:>8} | {err:>6.1f}%')
    else:
        print(f'{name:<15} | {chars:>6} | {char_est:>10} | {"N/A":>8} | {"N/A":>7}')

### Quality vs Compression Ratio

There is no free lunch — aggressive compression always risks removing relevant information.

| Ratio | Tokens kept | Quality impact | When to use |
|---|---|---|---|
| 1.0 (no compression) | 100% | None | Debugging, high-stakes queries |
| 0.8 | 80% | Negligible | Remove clearly irrelevant boilerplate |
| 0.5 | 50% | Moderate | Good default for multi-doc RAG |
| 0.3 | 30% | Noticeable | Useful when context window is the constraint |
| 0.1 | 10% | High | Extreme budget mode — accept quality drop |

> **Practical tip:** Start at 0.5 and measure answer quality against a holdout set.
> Only go lower if the context window or cost forces it.

In [ ]:
# Compress and merge multiple documents for a single query
from typing import List

def compress_multi_doc(docs: List[str], query: str,
                        per_doc_ratio: float = 0.4,
                        max_total_sentences: int = 20) -> str:
    """
    Compress each document independently, then merge and re-rank
    to stay within max_total_sentences across all documents.
    """
    import re, math

    def split_sentences(text):
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 10]

    def mock_score(sentence, query):
        # Mock: score = fraction of query words in sentence (lowercase)
        q_words = set(query.lower().split())
        s_words = set(sentence.lower().split())
        return len(q_words & s_words) / len(q_words) if q_words else 0.5

    all_scored = []
    for doc_idx, doc in enumerate(docs):
        sentences = split_sentences(doc)
        for s in sentences:
            all_scored.append((mock_score(s, query), doc_idx, s))

    all_scored.sort(reverse=True)
    top = all_scored[:max_total_sentences]
    # Restore reading order within each doc
    top.sort(key=lambda x: (x[1], docs[x[1]].find(x[2])))
    return '\n'.join(s for _, _, s in top)

DOCS = [
    'Python is a high-level language. It supports OOP and functional programming. Python is widely used in data science.',
    'Java is statically typed. It runs on the JVM. Java is used in enterprise applications and Android development.',
    'JavaScript runs in browsers. Node.js brought JavaScript to servers. It is the language of the web.',
]
result = compress_multi_doc(DOCS, query='which language is used for data science?', max_total_sentences=4)
print('Compressed multi-doc context:')
print(result)

In [ ]:
# Part A: ratio sweep with token counting (shows the actual token savings)
import re

SAMPLE_DOC = '''
Python is a high-level, general-purpose programming language.
Its design philosophy emphasizes code readability and simplicity.
Python is dynamically typed and garbage-collected.
It supports multiple programming paradigms including structured, OOP, and functional.
Guido van Rossum created Python and first released it in 1991.
Python consistently ranks as one of the most popular programming languages.
The language's design makes it popular for rapid application development.
Python has a comprehensive standard library often described as batteries-included.
It is commonly used in web development, scientific computing, AI, and scripting.
Python 3.0, released in 2008, was a major revision not fully backward-compatible with Python 2.
'''.strip()

sentences = [s.strip() for s in re.split(r'(?<=[.])\s+', SAMPLE_DOC) if s.strip()]
# Mock scores based on relevance to 'What makes Python popular?'
mock_scores = [0.9, 0.8, 0.5, 0.6, 0.4, 0.95, 0.85, 0.7, 0.88, 0.3]

QUERY = 'What makes Python popular for programming?'
print(f'Original: {len(sentences)} sentences, ~{sum(len(s) for s in sentences)//4} tokens')
print(f'Query: {QUERY}\n')

print(f'{'Ratio':>7} | {'Sentences':>9} | {'Est. tokens':>12} | {'Token savings':>14}')
print('-' * 50)
orig_tokens = sum(len(s)//4 for s in sentences)
for ratio in [1.0, 0.8, 0.6, 0.4, 0.2]:
    keep_n = max(1, int(len(sentences) * ratio))
    scored = sorted(zip(mock_scores, sentences), reverse=True)
    kept = {s for _, s in scored[:keep_n]}
    compressed = [s for s in sentences if s in kept]
    tok = sum(len(s)//4 for s in compressed)
    savings = (orig_tokens - tok) / orig_tokens * 100
    print(f'{ratio:>7.1f} | {len(compressed):>9} | {tok:>12} | {savings:>13.0f}%')

### Answer Quality vs Compression

Compression removes sentences — but which sentences are removed matters.
A well-calibrated relevance scorer should remove **background/boilerplate** first
and keep **direct evidence** for the query.

**Common failure modes:**
- Low-threshold scores treat all sentences as equally relevant → no compression effect
- High-threshold scores remove everything → model answers 'I don't know'
- Query too broad ('tell me about Python') → many sentences score equally, random selection

**Mitigation:** Always evaluate compression quality with a small holdout set
before deploying at scale. Compare answers from full context vs compressed context.

---
## What's Next

- **LLMLingua** (github.com/microsoft/LLMLingua) — token-level compression using a small LM as the scoring model; 3–20× compression with minimal quality loss
- **Selective-Context** (Li et al. 2023) — another extractive approach that uses PPL (perplexity) as the relevance signal
- **Example 141 — Prompt Caching:** once you've compressed the context, cache it so repeated queries don't re-score
- **Example 142 — Semantic Caching:** pair with semantic caching — compress before embedding, then cache the compressed+embedded form
- **RAG re-ranking:** use a cross-encoder reranker (e.g. Cohere Rerank) as the scoring function for higher accuracy